# 1.6. Scrape BGT Objects + Bomen Atlas + OSM

Extends **1.5 Scrape BGT Objects** by also fetching tree data from the **Amsterdam Bomen Atlas** (`/v1/bomen/stamgegevens/`) and three OpenStreetMap layers via the **Overpass API**.

Output files written to:
- `data/input/bgt/bgt_poles.csv` — BGT poles (boom, lichtmast, …)
- `data/input/bgt/bgt_street_furniture.csv` — BGT street furniture
- `data/input/bomen/bomen_atlas.csv` — trees from the Bomen Atlas
- `data/input/afvalbakken/afvalbakken_oor.csv` — municipal waste bins (OOR)
- `data/input/osm/osm_fietsparkeren.csv` — OSM bike parking (`amenity=bicycle_parking`)
- `data/input/osm/osm_parkeermeters.csv` — OSM parking meters (`amenity=parking_meter`)
- `data/input/osm/osm_reclameborden.csv` — OSM advertising signs (`advertising=*`)

## Config

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))

from config import BBOX_DIR, BGT_DIR, BOMEN_DIR, SETUP_TILECODES

tilecodes = SETUP_TILECODES

BOMEN_DIR.mkdir(parents=True, exist_ok=True)

## Helpers: bounding box + coordinate conversion

In [ ]:
import json
import numpy as np

def _geojson_bounds(path):
    with open(path) as f:
        data = json.load(f)
    xs, ys = [], []
    for feat in data.get("features", []):
        geom = feat.get("geometry") or {}
        gtype = geom.get("type", "")
        if gtype == "Polygon":
            for ring in geom["coordinates"]:
                for x, y in ring:
                    xs.append(x); ys.append(y)
        elif gtype == "MultiPolygon":
            for poly in geom["coordinates"]:
                for ring in poly:
                    for x, y in ring:
                        xs.append(x); ys.append(y)
    return min(xs), min(ys), max(xs), max(ys)

padding = 10  # metres
x_min = y_min = 1e9
x_max = y_max = -1e9

for tc in tilecodes:
    bx0, by0, bx1, by1 = _geojson_bounds(BBOX_DIR / f"bbox_{tc}.geojson")
    x_min = min(x_min, bx0); y_min = min(y_min, by0)
    x_max = max(x_max, bx1); y_max = max(y_max, by1)

x_min -= padding; y_min -= padding
x_max += padding; y_max += padding

print(f"Scrape area  x: [{x_min:.0f}, {x_max:.0f}]")
print(f"             y: [{y_min:.0f}, {y_max:.0f}]")
print(f"Area: {(x_max-x_min)*(y_max-y_min)/1e4:.1f} ha")

In [ ]:
# RD New <-> WGS84 (inline, no extra dependencies)
_X0, _Y0 = 155000.0, 463000.0
_PHI0, _LAM0 = 52.15517440, 5.38720621

_K = [(0,1,3235.65389),(2,0,-32.58297),(0,2,-0.24750),(2,1,-0.84978),
      (0,3,-0.06550),(2,2,0.01709),(1,0,-0.00738),(4,0,0.00530),
      (2,3,-0.00039),(4,1,0.00033),(1,1,-0.00012)]
_L = [(1,0,5260.52916),(1,1,105.94684),(1,2,2.45656),(3,0,-0.81885),
      (1,3,0.05594),(3,1,-0.05607),(0,1,0.01199),(3,2,0.00256),
      (1,4,0.00128),(0,2,0.00022),(2,0,-0.00022),(5,0,0.00026)]
_M = [(0,1,0.336748),(2,0,-0.014675),(0,2,-0.001907),(2,1,-0.000453),(0,3,0.000044)]
_N = [(1,0,0.597498),(1,1,0.036633),(1,2,0.002661),(3,0,-0.003495),
      (1,3,-0.000063),(3,1,-0.001060)]

def rd_to_wgs84(x, y):
    dx = (x - _X0) * 1e-5; dy = (y - _Y0) * 1e-5
    phi = _PHI0 + sum(p * dx**a * dy**b for a,b,p in _K) / 3600
    lam = _LAM0 + sum(p * dx**a * dy**b for a,b,p in _L) / 3600
    return phi, lam

def wgs84_to_rd(lat, lon):
    dphi = 0.36*(lat-_PHI0); dlam = 0.36*(lon-_LAM0)
    x = _X0 + sum(p*dphi**a*dlam**b for a,b,p in _N)*1e5
    y = _Y0 + sum(p*dphi**a*dlam**b for a,b,p in _M)*1e5
    return x, y

def rd_bbox_to_wgs84_wkt(x0, y0, x1, y1):
    """Return a closed WGS84 WKT POLYGON (lon lat) from an RD New bounding box."""
    corners = [(x0, y0), (x1, y0), (x1, y1), (x0, y1), (x0, y0)]
    parts = []
    for x, y in corners:
        lat, lon = rd_to_wgs84(x, y)
        parts.append(f"{lon:.7f} {lat:.7f}")
    return "POLYGON((" + ", ".join(parts) + "))"

wkt_scrape_area = rd_bbox_to_wgs84_wkt(x_min, y_min, x_max, y_max)
print("WKT ready.")

## Shared fetch helper

In [ ]:
import time, csv, urllib.parse, urllib.request

def _fetch_url(url, timeout=90, retries=3, pause=2.0):
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as resp:
                return json.loads(resp.read())
        except urllib.error.URLError as exc:
            if attempt < retries - 1:
                print(f"    attempt {attempt+1} failed ({exc}), retrying ...", flush=True)
                time.sleep(pause * (attempt + 1))
            else:
                raise

def _fetch_all(base_url, embed_key, page_size=1000, pause=0.3):
    """Page through a DSO REST API endpoint and return all embedded items."""
    url = base_url + f"&_pageSize={page_size}&_format=json"
    items = []
    while url:
        data = _fetch_url(url)
        items.extend(data.get("_embedded", {}).get(embed_key, []))
        next_link = (data.get("_links") or {}).get("next") or {}
        url = next_link.get("href") if isinstance(next_link, dict) else None
        if url:
            time.sleep(pause)
    return items

## Scrape BGT objects (same as 1.5)

In [ ]:
BGT_BASE = "https://api.data.amsterdam.nl/v1/bgt/"

POINT_LAYERS = {
    "lichtmast":                      ("palen",             "palen"),
    "verkeersbord":                   ("palen",             "palen"),
    "verkeersregelinstallatiepaal":   ("palen",             "palen"),
    "verkeersbordpaal":               ("palen",             "palen"),
    "poller":                         ("palen",             "palen"),
    "haltepaal":                      ("palen",             "palen"),
    "bank":                           ("straatmeubilair",   "straatmeubilair"),
    "afvalbak":                       ("bakken",            "bakken"),
    "fietsenrek":                     ("straatmeubilair",   "straatmeubilair"),
    "afval apart plaats":             ("bakken",            "bakken"),
}

POLES_TYPES     = ["lichtmast", "verkeersbord",
                   "verkeersregelinstallatiepaal", "verkeersbordpaal",
                   "poller", "haltepaal"]
FURNITURE_TYPES = ["bank", "afvalbak", "fietsenrek", "afval apart plaats"]

poly_enc = urllib.parse.quote(wkt_scrape_area)

def scrape_bgt_type(bgt_type, endpoint, embed_key):
    base = (f"{BGT_BASE}{endpoint}/?plusType={urllib.parse.quote(bgt_type)}"
            f"&geometrie[intersects]={poly_enc}")
    items = _fetch_all(base, embed_key)
    rows = []
    for item in items:
        geom = item.get("geometrie") or item.get("geometriePunt") or {}
        if geom.get("type") != "Point":
            continue
        x_rd, y_rd = geom["coordinates"][0], geom["coordinates"][1]
        rows.append([bgt_type, round(x_rd, 3), round(y_rd, 3)])
    return rows

poles_rows = []
furniture_rows = []

for bgt_type, (endpoint, embed_key) in POINT_LAYERS.items():
    print(f"  scraping BGT {bgt_type} ...", end=" ", flush=True)
    rows = scrape_bgt_type(bgt_type, endpoint, embed_key)
    print(f"{len(rows)} objects")
    if bgt_type in FURNITURE_TYPES:
        furniture_rows.extend(rows)
    else:
        poles_rows.extend(rows)

print(f"\nTotal poles:     {len(poles_rows)}")
print(f"Total furniture: {len(furniture_rows)}")

## Scrape Bomen Atlas

Fetches trees from `/v1/bomen/stamgegevens/` which is the source behind [bomen.amsterdam.nl](https://bomen.amsterdam.nl).
Each record includes species, height class, planting year, and an RD New point geometry.

In [ ]:
BOMEN_BASE = "https://api.data.amsterdam.nl/v1/bomen/stamgegevens/"

print("Scraping Bomen Atlas ...", end=" ", flush=True)
bomen_items = _fetch_all(
    f"{BOMEN_BASE}?geometrie[intersects]={poly_enc}",
    embed_key="stamgegevens",
)
print(f"{len(bomen_items)} trees")

bomen_rows = []
for item in bomen_items:
    geom = item.get("geometrie") or {}
    if geom.get("type") != "Point":
        continue
    x_rd, y_rd = geom["coordinates"][0], geom["coordinates"][1]
    bomen_rows.append({
        "id":                   item.get("id"),
        "x":                    round(x_rd, 3),
        "y":                    round(y_rd, 3),
        # "soortnaam":            item.get("soortnaam"),
        # "soortnaamKort":        item.get("soortnaamKort"),
        # "soortnaamTop":         item.get("soortnaamTop"),
        # "boomhoogteklasse":     item.get("boomhoogteklasseActueel"),
        # "stamdiameterklasse":   item.get("stamdiameterklasse"),
        # "jaarVanAanleg":        item.get("jaarVanAanleg"),
        # "standplaats":          item.get("standplaats"),
        # "typeObject":           item.get("typeObject"),
    })

print(f"Parsed {len(bomen_rows)} point records")

## Save to CSV

In [ ]:
import pandas as pd

BGT_DIR.mkdir(parents=True, exist_ok=True)

def write_csv(path, rows, fieldnames=None):
    if not rows:
        print(f"  (no rows to write) -> {path}")
        return
    if fieldnames is None:
        fieldnames = list(rows[0].keys()) if isinstance(rows[0], dict) else None
    with open(path, "w", newline="") as f:
        if fieldnames:
            w = csv.DictWriter(f, fieldnames=fieldnames)
            w.writeheader(); w.writerows(rows)
        else:
            w = csv.writer(f)
            w.writerow(["bgt_type", "x", "y"]); w.writerows(rows)
    print(f"  Saved {len(rows):,} rows -> {path}")

write_csv(BGT_DIR  / "bgt_poles.csv",          poles_rows)
write_csv(BGT_DIR  / "bgt_street_furniture.csv", furniture_rows)
write_csv(BOMEN_DIR / "bomen_atlas.csv",         bomen_rows)

## Scrape afvalbakken (objectenopenbareruimte)

The BGT `afvalbak` type misses most regular streetside bins. The `objectenopenbareruimte/afvalbakken/` dataset is Amsterdam's municipal asset management inventory (GISIB) and has ~10k bins citywide with model name, capacity, and management area.

Output saved to `data/input/afvalbakken/afvalbakken_oor.csv`.

In [ ]:
from config import AFVAL_DIR

OOR_BASE = "https://api.data.amsterdam.nl/v1/objectenopenbareruimte/"
AFVAL_DIR.mkdir(parents=True, exist_ok=True)

print("Scraping afvalbakken (objectenopenbareruimte) ...", end=" ", flush=True)
oor_items = _fetch_all(
    f"{OOR_BASE}afvalbakken/?geometrie[intersects]={poly_enc}",
    embed_key="afvalbakken",
)
print(f"{len(oor_items)} bins")

oor_rows = []
for item in oor_items:
    geom = item.get("geometrie") or {}
    if geom.get("type") != "Point":
        continue
    x_rd, y_rd = geom["coordinates"][0], geom["coordinates"][1]
    oor_rows.append({
        "id":           item.get("id"),
        "x":            round(x_rd, 3),
        "y":            round(y_rd, 3),
        # "naam":         item.get("naam"),
        # "type":         item.get("type"),
        # "inhoud":       item.get("inhoud"),
        # "actief":       item.get("actief"),
        # "bouwjaar":     item.get("bouwjaar"),
        # "beheerder":    item.get("beheerder"),
        # "straatnaam":   item.get("straatnaam"),
        # "huisnummer":   item.get("huisnummer"),
        # "buurtNaam":    item.get("buurtNaam"),
        # "wijkNaam":     item.get("wijkNaam"),
        # "stadsdeelNaam":item.get("stadsdeelNaam"),
    })

write_csv(AFVAL_DIR / "afvalbakken_oor.csv", oor_rows)

## Scrape OSM: fietsparkeren, parkeermeters & reclameborden

Fetches three point layers from OpenStreetMap via the **Overpass API**:

| Layer | OSM tag | Output file |
|---|---|---|
| Fietsparkeren | `amenity=bicycle_parking` | `osm_fietsparkeren.csv` |
| Parkeermeters | `amenity=parking_meter` | `osm_parkeermeters.csv` |
| Reclameborden | `advertising=*` | `osm_reclameborden.csv` |

All coordinates are returned in WGS84 and converted to RD New (x/y) to match the rest of the pipeline.

In [ ]:
from config import OSM_DIR
OSM_DIR.mkdir(parents=True, exist_ok=True)

# Bounding box in WGS84 for Overpass API (south, west, north, east)
lat_sw, lon_sw = rd_to_wgs84(x_min, y_min)
lat_ne, lon_ne = rd_to_wgs84(x_max, y_max)
overpass_bbox  = f"{lat_sw:.7f},{lon_sw:.7f},{lat_ne:.7f},{lon_ne:.7f}"
print(f"Overpass bbox (S,W,N,E): {overpass_bbox}")

OVERPASS_URL = "https://overpass-api.de/api/interpreter"

def fetch_overpass(query):
    payload = urllib.parse.urlencode({"data": query}).encode("utf-8")
    req = urllib.request.Request(OVERPASS_URL, data=payload)
    req.add_header("Content-Type", "application/x-www-form-urlencoded")
    req.add_header("Accept", "*/*")
    req.add_header("User-Agent", "overpass-scraper/1.0")
    with urllib.request.urlopen(req, timeout=120) as resp:
        return json.loads(resp.read()).get("elements", [])

def overpass_to_rows(elements, osm_type):
    rows = []
    for el in elements:
        if el["type"] == "node":
            lat, lon = el["lat"], el["lon"]
        elif "center" in el:
            lat, lon = el["center"]["lat"], el["center"]["lon"]
        else:
            continue
        x_rd, y_rd = wgs84_to_rd(lat, lon)
        tags = el.get("tags", {})
        rows.append({
            "osm_id":          el["id"],
            "osm_type":        el["type"],
            "x":               round(x_rd, 3),
            "y":               round(y_rd, 3),
            "type":            osm_type,
            "amenity":         tags.get("amenity", ""),
            "bicycle_parking": tags.get("bicycle_parking", ""),
            "capacity":        tags.get("capacity", ""),
            "covered":         tags.get("covered", ""),
            "advertising":     tags.get("advertising", ""),
            "name":            tags.get("name", ""),
            "operator":        tags.get("operator", ""),
        })
    return rows

OSM_QUERIES = {
    "fietsparkeren": f"""
[out:json][timeout:60];
(
  node["amenity"="bicycle_parking"]({overpass_bbox});
  way["amenity"="bicycle_parking"]({overpass_bbox});
);
out center tags;
""",
    "parkeermeters": f"""
[out:json][timeout:60];
(
  node["amenity"="parking_meter"]({overpass_bbox});
);
out center tags;
""",
    "reclameborden": f"""
[out:json][timeout:60];
(
  node["advertising"]({overpass_bbox});
  way["advertising"]({overpass_bbox});
);
out center tags;
""",
}

_OSM_FIELDNAMES = ["osm_id", "osm_type", "x", "y", "type", "amenity",
                   "bicycle_parking", "capacity", "covered", "advertising",
                   "name", "operator"]

osm_results = {}
for name, query in OSM_QUERIES.items():
    print(f"  scraping OSM {name} ...", end=" ", flush=True)
    elements = fetch_overpass(query)
    rows = overpass_to_rows(elements, name)
    osm_results[name] = rows
    write_csv(OSM_DIR / f"osm_{name}.csv", rows, fieldnames=_OSM_FIELDNAMES)

print(f"\nOSM fietsparkeren: {len(osm_results['fietsparkeren'])}")
print(f"OSM parkeermeters: {len(osm_results['parkeermeters'])}")
print(f"OSM reclameborden: {len(osm_results['reclameborden'])}")

## Scrape terrassen (exploitatievergunning-terrasgeometrie)

Fetches **terras polygons** from the Amsterdam horeca WFS endpoint:
`/v1/wfs/horeca/` → layer `exploitatievergunning-terrasgeometrie`

Each feature is a **MultiPolygon** in RD New covering the licensed outdoor seating footprint.
~2,400 terrassen citywide have a geometry; total dataset is ~4,200 records.

Output:
- `data/input/terras/terrassen.geojson` — full polygon features
- `data/input/terras/terrassen_centroids.csv` — centroid per terras for point-based analysis


In [ ]:
from config import TERRAS_DIR
TERRAS_DIR.mkdir(parents=True, exist_ok=True)

TERRAS_WFS = "https://api.data.amsterdam.nl/v1/wfs/horeca/"
TERRAS_LAYER = "app:exploitatievergunning-terrasgeometrie"

def fetch_wfs_page(layer, start_index, count=1000):
    params = urllib.parse.urlencode({
        "SERVICE": "WFS", "VERSION": "2.0.0", "REQUEST": "GetFeature",
        "TYPENAMES": layer, "OUTPUTFORMAT": "application/json",
        "COUNT": count, "STARTINDEX": start_index,
    })
    return _fetch_url(f"{TERRAS_WFS}?{params}")

print("Scraping terras polygons (WFS) ...", flush=True)
terras_features = []
start = 0
while True:
    data = fetch_wfs_page(TERRAS_LAYER, start)
    page = data.get("features", [])
    if not page:
        break
    with_geom = [f for f in page if f.get("geometry") is not None]
    terras_features.extend(with_geom)
    print(f"  offset {start}: {len(page)} records, {len(with_geom)} with geometry")
    if len(page) < 1000:
        break
    start += 1000
    time.sleep(0.3)

print(f"\nTotal terras polygons: {len(terras_features)}")

# Save GeoJSON (polygons)
geojson_out = {
    "type": "FeatureCollection",
    "crs": {"type": "name", "properties": {"name": "urn:ogc:def:crs:EPSG::28992"}},
    "features": terras_features,
}
with open(TERRAS_DIR / "terrassen.geojson", "w") as f:
    json.dump(geojson_out, f)
print(f"Saved -> {TERRAS_DIR}/terrassen.geojson")

# Save centroid CSV for point-based analysis
def _poly_centroid(geom):
    coords = []
    if geom["type"] == "MultiPolygon":
        for poly in geom["coordinates"]:
            for ring in poly: coords.extend(ring)
    elif geom["type"] == "Polygon":
        for ring in geom["coordinates"]: coords.extend(ring)
    return (round(sum(c[0] for c in coords)/len(coords), 3),
            round(sum(c[1] for c in coords)/len(coords), 3)) if coords else (None, None)

terras_rows = []
for feat in terras_features:
    cx, cy = _poly_centroid(feat["geometry"])
    p = feat.get("properties", {})
    terras_rows.append({
        "id":               p.get("id"),
        "x":                cx,
        "y":                cy,
        "zaaknaam":         p.get("zaaknaam"),
        "adres":            p.get("adres"),
        "zaak_categorie":   p.get("zaak_categorie"),
        "status_vergunning":p.get("status_vergunning"),
        "begindatum":       p.get("begindatum"),
        "einddatum":        p.get("einddatum"),
    })

write_csv(TERRAS_DIR / "terrassen_centroids.csv", terras_rows,
          fieldnames=["id","x","y","zaaknaam","adres","zaak_categorie",
                      "status_vergunning","begindatum","einddatum"])

In [ ]:
TERRAS_WFS   = "https://api.data.amsterdam.nl/v1/wfs/horeca/"
TERRAS_LAYER = "app:exploitatievergunning-terrasgeometrie"

# Fetch terrassen intersecting the scrape area (tile bounding box)
bbox_rd = f"{x_min:.3f},{y_min:.3f},{x_max:.3f},{y_max:.3f},urn:ogc:def:crs:EPSG::28992"

print(f"Fetching terrassen for tiles: {tilecodes}", flush=True)
params = urllib.parse.urlencode({
    "SERVICE": "WFS", "VERSION": "2.0.0", "REQUEST": "GetFeature",
    "TYPENAMES": TERRAS_LAYER, "OUTPUTFORMAT": "application/json",
    "BBOX": bbox_rd,
})
data = _fetch_url(f"{TERRAS_WFS}?{params}")
terras_tile_features = [f for f in data.get("features", []) if f.get("geometry") is not None]
print(f"Terrassen in scrape area: {len(terras_tile_features)}")

if terras_tile_features:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 10))

    # Tile outlines
    for tc in tilecodes:
        with open(BBOX_DIR / f"bbox_{tc}.geojson") as f:
            gjson = json.load(f)
        for feat in gjson["features"]:
            ring = feat["geometry"]["coordinates"][0]
            xs, ys = zip(*ring)
            ax.plot(xs, ys, "k-", linewidth=1.5, zorder=3)

    # Terras polygons
    for feat in terras_tile_features:
        geom = feat["geometry"]
        p    = feat.get("properties", {})
        rings = []
        if geom["type"] == "MultiPolygon":
            for poly in geom["coordinates"]: rings.extend(poly)
        elif geom["type"] == "Polygon":
            rings = geom["coordinates"]
        for ring in rings:
            ax.fill([c[0] for c in ring], [c[1] for c in ring],
                    color="#f4d03f", alpha=0.6, zorder=2)
            ax.plot([c[0] for c in ring], [c[1] for c in ring],
                    color="#d4ac0d", linewidth=0.8, zorder=2)
        # Label with name
        cx, cy = _poly_centroid(geom)
        if cx:
            ax.annotate(p.get("zaaknaam", ""), (cx, cy),
                        fontsize=6, ha="center", zorder=4,
                        xytext=(0, 3), textcoords="offset points")

    ax.set_aspect("equal")
    ax.set_xlabel("X (m RD New)")
    ax.set_ylabel("Y (m RD New)")
    ax.set_title(f"Terrassen in scrape area — {len(terras_tile_features)} polygons\n"
                 f"tiles: {', '.join(tilecodes)}")
    plt.tight_layout()
    plt.show()


## Map: all scraped objects

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

COLORS = {
    "lichtmast":          ("#ff7f0e", "Lamp posts",          15),
    "bank":               ("#1f77b4", "Benches",             25),
    "afvalbak":           ("#9467bd", "BGT Bins",            20),
    "afval apart plaats": ("#8c564b", "BGT Containers",      25),
    "fietsenrek":         ("#17becf", "Bike racks",          25),
}

bgt_all  = pd.DataFrame(poles_rows + furniture_rows, columns=["bgt_type", "x", "y"])
bomen_df = pd.DataFrame(bomen_rows)
oor_df   = pd.DataFrame(oor_rows)

# terras_tile_features is set by the terras-tile cell; fall back to filtering from the full set
if "terras_tile_features" not in dir():
    terras_tile_features = [
        f for f in terras_features
        if any(
            x_min <= cx <= x_max and y_min <= cy <= y_max
            for poly in (f["geometry"]["coordinates"] if f["geometry"]["type"] == "MultiPolygon"
                         else [f["geometry"]["coordinates"]])
            for ring in poly
            for cx, cy in ring
        )
    ]

fig, ax = plt.subplots(figsize=(12, 12))

# Terras polygons in tile area only
for feat in terras_tile_features:
    geom = feat["geometry"]
    rings = []
    if geom["type"] == "MultiPolygon":
        for poly in geom["coordinates"]: rings.extend(poly)
    elif geom["type"] == "Polygon":
        rings = geom["coordinates"]
    for ring in rings:
        ax.fill([c[0] for c in ring], [c[1] for c in ring],
                color="#f4d03f", alpha=0.4, zorder=2)
        ax.plot([c[0] for c in ring], [c[1] for c in ring],
                color="#d4ac0d", linewidth=0.5, zorder=2)

# Tile outlines
for tc in tilecodes:
    with open(BBOX_DIR / f"bbox_{tc}.geojson") as f:
        gjson = json.load(f)
    for feat in gjson["features"]:
        ring = feat["geometry"]["coordinates"][0]
        xs, ys = zip(*ring)
        ax.plot(xs, ys, "k-", linewidth=1.5, zorder=3)

# BGT objects (excluding boom — trees come from Bomen Atlas only)
for bgt_type, (colour, label, size) in COLORS.items():
    sub = bgt_all[bgt_all["bgt_type"] == bgt_type]
    if len(sub):
        ax.scatter(sub["x"], sub["y"], c=colour, s=size, label=f"{label} ({len(sub)})",
                   zorder=4, linewidths=0)

# Bomen Atlas trees
if len(bomen_df):
    ax.scatter(bomen_df["x"], bomen_df["y"], c="#2ca02c", s=35, marker="*",
               label=f"Bomen Atlas ({len(bomen_df)})", zorder=5, linewidths=0, alpha=0.7)

# OOR afvalbakken
if len(oor_df):
    ax.scatter(oor_df["x"], oor_df["y"], c="#e377c2", s=25, marker="s",
               label=f"Afvalbakken OOR ({len(oor_df)})", zorder=5, linewidths=0)

# OSM layers
OSM_STYLE = {
    "fietsparkeren": ("#0055ff", "^", 40, "OSM fietsparkeren"),
    "parkeermeters": ("#cc00cc", "D", 30, "OSM parkeermeters"),
    "reclameborden": ("#ff4500", "P", 40, "OSM reclameborden"),
}
for name, (colour, marker, size, label) in OSM_STYLE.items():
    rows = osm_results.get(name, [])
    if rows:
        xs = [r["x"] for r in rows]
        ys = [r["y"] for r in rows]
        ax.scatter(xs, ys, c=colour, s=size, marker=marker,
                   label=f"{label} ({len(rows)})", zorder=6, linewidths=0, alpha=0.85)

# Legend entry for terrassen
terras_patch = mpatches.Patch(facecolor="#f4d03f", edgecolor="#d4ac0d", alpha=0.7,
                               label=f"Terrassen ({len(terras_tile_features)})")

ax.set_aspect("equal")
ax.set_xlabel("X (m RD New)")
ax.set_ylabel("Y (m RD New)")
ax.set_title(f"All scraped objects — tiles: {', '.join(tilecodes)}")
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=[terras_patch] + handles, labels=[terras_patch.get_label()] + labels,
          loc="upper right", framealpha=0.9, fontsize=9)
plt.tight_layout()
plt.show()


## Compare BGT trees vs Bomen Atlas

For each BGT `boom` point, find the nearest Bomen Atlas tree. Then classify each pair as:
- **matched** — nearest Bomen Atlas tree is within `match_radius` metres
- **BGT only** — BGT tree with no close Bomen Atlas counterpart
- **Bomen only** — Bomen Atlas tree with no close BGT counterpart

In [ ]:
import numpy as np

match_radius = 3.0  # metres — trees closer than this are considered the same tree

# Build arrays
bgt_trees = np.array([[r[1], r[2]] for r in poles_rows if r[0] == "boom"])
bomen_xy  = np.array([[r["x"], r["y"]] for r in bomen_rows])

print(f"BGT boom points:    {len(bgt_trees)}")
print(f"Bomen Atlas points: {len(bomen_xy)}")

# For every BGT tree find the nearest Bomen Atlas tree
def nearest_distances(src, tgt):
    """Return (min_dist, nearest_idx) for each point in src against tgt."""
    if len(tgt) == 0:
        return np.full(len(src), np.inf), np.full(len(src), -1, dtype=int)
    dists = np.linalg.norm(src[:, None, :] - tgt[None, :, :], axis=2)
    idx   = dists.argmin(axis=1)
    return dists[np.arange(len(src)), idx], idx

bgt_min_dist, bgt_nearest = nearest_distances(bgt_trees, bomen_xy)
bomen_min_dist, _          = nearest_distances(bomen_xy, bgt_trees)

bgt_matched  = bgt_min_dist   <= match_radius
bomen_matched = bomen_min_dist <= match_radius

n_matched    = bgt_matched.sum()
n_bgt_only   = (~bgt_matched).sum()
n_bomen_only = (~bomen_matched).sum()

print(f"\nMatch radius: {match_radius} m")
print(f"  Matched (in both):  {n_matched}")
print(f"  BGT only (missing from Bomen Atlas): {n_bgt_only}")
print(f"  Bomen only (missing from BGT):       {n_bomen_only}")

## Map: BGT trees vs Bomen Atlas

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(11, 11))

# Tile polygon outlines
for tc in tilecodes:
    with open(BBOX_DIR / f"bbox_{tc}.geojson") as f:
        gjson = json.load(f)
    for feat in gjson["features"]:
        ring = feat["geometry"]["coordinates"][0]
        xs, ys = zip(*ring)
        ax.plot(xs, ys, "k-", linewidth=1.5, zorder=2)

# Bomen Atlas trees (background layer)
if len(bomen_xy):
    ax.scatter(bomen_xy[:, 0], bomen_xy[:, 1],
               c="#2ca02c", s=30, marker="o", linewidths=0,
               label=f"Bomen Atlas ({len(bomen_xy)})", zorder=3, alpha=0.6)

# BGT trees — colour by match status
if len(bgt_trees):
    matched_pts  = bgt_trees[bgt_matched]
    only_pts     = bgt_trees[~bgt_matched]
    if len(matched_pts):
        ax.scatter(matched_pts[:, 0], matched_pts[:, 1],
                   c="#1f77b4", s=50, marker="^", linewidths=0.5,
                   edgecolors="white", label=f"BGT matched ({len(matched_pts)})",
                   zorder=5)
    if len(only_pts):
        ax.scatter(only_pts[:, 0], only_pts[:, 1],
                   c="#d62728", s=60, marker="^", linewidths=0.5,
                   edgecolors="white", label=f"BGT only / no Bomen match ({len(only_pts)})",
                   zorder=5)

# Bomen-only trees (no BGT counterpart)
bomen_only_pts = bomen_xy[~bomen_matched]
if len(bomen_only_pts):
    ax.scatter(bomen_only_pts[:, 0], bomen_only_pts[:, 1],
               c="#ff7f0e", s=30, marker="o", linewidths=0,
               label=f"Bomen only / missing from BGT ({len(bomen_only_pts)})",
               zorder=4, alpha=0.9)

ax.set_aspect("equal")
ax.set_xlabel("X (m RD New)")
ax.set_ylabel("Y (m RD New)")
ax.set_title(f"BGT boom vs Bomen Atlas  —  match radius {match_radius} m\n"
             f"tiles: {', '.join(tilecodes)}")
ax.legend(loc="upper right", framealpha=0.9, fontsize=9)
plt.tight_layout()
plt.show()

## Distance distribution for matched pairs

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(bgt_min_dist, bins=40, color="#1f77b4", edgecolor="white", linewidth=0.4)
ax.axvline(match_radius, color="red", linestyle="--", label=f"match radius ({match_radius} m)")
ax.set_xlabel("Distance to nearest Bomen Atlas tree (m)")
ax.set_ylabel("BGT boom count")
ax.set_title("How far is each BGT boom from the nearest Bomen Atlas tree?")
ax.legend()
plt.tight_layout()
plt.show()

## Diagnostic: OOR afvalbakken vs 2D obstacles

For each OOR afvalbak in tile `120300_489300`, find the nearest 2D obstacle polygon and check whether the bin falls inside or close to one. This tells us whether the bin exists as a detected obstacle at all — before the labeling step tries to match it.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection
import pandas as pd

FOCUS_TILE   = "120300_489300"
OBSTACLE_DIR = Path("data/output/obstacles")
SNAP_RADIUS  = 2.0   # metres — highlight obstacles within this distance of a bin

# --- load OOR afvalbakken for the focus tile ---
oor_df = pd.read_csv(AFVAL_DIR / "afvalbakken_oor.csv")
tile_bbox = _geojson_bounds(BBOX_DIR / f"bbox_{FOCUS_TILE}.geojson")
bx0, by0, bx1, by1 = tile_bbox
bins_in_tile = oor_df[
    (oor_df.x >= bx0) & (oor_df.x <= bx1) &
    (oor_df.y >= by0) & (oor_df.y <= by1)
].reset_index(drop=True)
print(f"OOR afvalbakken in {FOCUS_TILE}: {len(bins_in_tile)}")
print(bins_in_tile[["id", "x", "y", "naam", "straatnaam"]].to_string())

# --- load 2D obstacles ---
with open(OBSTACLE_DIR / f"obstacles_2d_{FOCUS_TILE}.geojson") as f:
    obs_gj = json.load(f)
obstacles = obs_gj["features"]
print(f"\n2D obstacles in tile: {len(obstacles)}")

# --- compute centroid of each obstacle polygon ---
def poly_centroid(ring):
    xs = [c[0] for c in ring]
    ys = [c[1] for c in ring]
    return np.mean(xs), np.mean(ys)

obs_centroids = np.array([
    poly_centroid(f["geometry"]["coordinates"][0]) for f in obstacles
])

# --- for each bin find nearest obstacle and its distance ---
bin_xy = bins_in_tile[["x", "y"]].values
print("\nNearest obstacle to each bin:")
near_obs_indices = []
for i, (bx, by) in enumerate(bin_xy):
    dists = np.linalg.norm(obs_centroids - np.array([bx, by]), axis=1)
    nearest_i = int(np.argmin(dists))
    nearest_d = dists[nearest_i]
    near_obs_indices.append(nearest_i if nearest_d <= SNAP_RADIUS else None)
    area = obstacles[nearest_i]["properties"]["area"]
    print(f"  bin {bins_in_tile.loc[i,'id']} ({bx:.1f}, {by:.1f})"
          f"  -> obstacle centroid {obs_centroids[nearest_i]}  dist={nearest_d:.2f} m  area={area:.2f} m²")

In [ ]:
# --- plot ---
fig, ax = plt.subplots(figsize=(11, 11))

# tile boundary
with open(BBOX_DIR / f"bbox_{FOCUS_TILE}.geojson") as f:
    tile_gj = json.load(f)
for feat in tile_gj["features"]:
    ring = feat["geometry"]["coordinates"][0]
    xs, ys = zip(*ring)
    ax.plot(xs, ys, "k-", linewidth=1.5, zorder=2)

# all 2D obstacles — grey fill
for i, feat in enumerate(obstacles):
    ring = np.array(feat["geometry"]["coordinates"][0])
    patch = MplPolygon(ring, closed=True)
    is_near = i in [j for j in near_obs_indices if j is not None]
    color = "#f4a261" if is_near else "#aaaaaa"
    ax.add_patch(plt.Polygon(ring, closed=True, facecolor=color,
                             edgecolor="#555555", linewidth=0.5,
                             alpha=0.7, zorder=3))

# OOR bin locations
for i, row in bins_in_tile.iterrows():
    matched = near_obs_indices[i] is not None
    color   = "#d62728" if matched else "#1f77b4"
    ax.scatter(row.x, row.y, c=color, s=120, marker="s",
               zorder=6, linewidths=0.8, edgecolors="white")
    ax.annotate(f"  {int(row.id)}\n  {row.naam}\n  d={np.linalg.norm(obs_centroids - np.array([row.x, row.y]), axis=1).min():.1f}m",
                (row.x, row.y), fontsize=7, zorder=7,
                xytext=(4, 4), textcoords="offset points")

# legend
legend_handles = [
    mpatches.Patch(facecolor="#aaaaaa", edgecolor="#555555", label="2D obstacle (no nearby bin)"),
    mpatches.Patch(facecolor="#f4a261", edgecolor="#555555", label=f"2D obstacle near a bin (≤{SNAP_RADIUS} m)"),
    mpatches.Patch(facecolor="#d62728", label=f"OOR bin matched to obstacle (≤{SNAP_RADIUS} m)"),
    mpatches.Patch(facecolor="#1f77b4", label=f"OOR bin — NO nearby obstacle (>{SNAP_RADIUS} m)"),
]
ax.legend(handles=legend_handles, loc="upper right", fontsize=8, framealpha=0.9)

ax.set_aspect("equal")
ax.set_xlabel("X (m RD New)")
ax.set_ylabel("Y (m RD New)")
ax.set_title(f"OOR afvalbakken vs 2D obstacles — {FOCUS_TILE}\n"
             f"Snap radius: {SNAP_RADIUS} m")
plt.tight_layout()
plt.show()